Journée Data Science

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_curve, roc_auc_score

In [ ]:
train_df = pd.read_csv('../data/farms_train.csv')
test_df = pd.read_csv('../data/farms_test.csv')

X = train_df.drop('DIFF', axis=1)
y = train_df['DIFF']

train_df
print(x)

,DIFF,TOF,AGE,R7,R8,R17,R22,R32
0,1,1,50,0.613,0.19960,0.0425,0.8637,0.2882
1,0,3,40,0.856,0.87970,0.0599,0.6241,0.0917
2,0,3,61,0.396,0.08729,0.0748,0.3028,0.3447
3,0,3,49,0.665,0.39470,0.1100,0.7716,0.2397
4,0,3,47,0.565,0.10440,0.0754,0.3295,0.2094
...,...,...,...,...,...,...,...,...
395,0,1,56,0.739,0.57450,0.0355,0.5530,0.1929
396,1,3,54,0.631,0.05324,0.0708,4.9711,0.1888
397,1,1,39,0.789,0.83510,0.0610,0.5920,0.2928
398,0,5,32,1.026,0.10040,0.0470,3.9397,0.1413


In [ ]:
# 2. Création d'un jeu de validation local (indispensable pour voir le score de notre côté)
# On garde 30% des données pour tester nos modèles (stratify=y garantit le même ratio de 0 et de 1)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 3. Normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# 4. Entraînement du SVM Rigide (C très grand)
svm_rigide = SVC(kernel='rbf', C=1000, random_state=42)
svm_rigide.fit(X_train_scaled, y_train)
# On utilise decision_function pour obtenir le score continu (nécessaire pour la courbe ROC)
scores_rigide = svm_rigide.decision_function(X_val_scaled)
auc_rigide = roc_auc_score(y_val, scores_rigide)

# 5. Entraînement du SVM Souple (C très petit)
svm_souple = SVC(kernel='rbf', C=0.1, random_state=42)
svm_souple.fit(X_train_scaled, y_train)
scores_souple = svm_souple.decision_function(X_val_scaled)
auc_souple = roc_auc_score(y_val, scores_souple)

# 6. Récupération des points pour tracer les courbes ROC
fpr_rigide, tpr_rigide, _ = roc_curve(y_val, scores_rigide)
fpr_souple, tpr_souple, _ = roc_curve(y_val, scores_souple)

# 7. Affichage des résultats et de la courbe ROC
print(f"AUC du SVM Rigide (C=1000) : {auc_rigide:.3f}")
print(f"AUC du SVM Souple (C=0.1)  : {auc_souple:.3f}")

plt.figure(figsize=(8, 6))
plt.plot(fpr_rigide, tpr_rigide, color='red', lw=2, 
         label=f'SVM Rigide (AUC = {auc_rigide:.3f})')
plt.plot(fpr_souple, tpr_souple, color='blue', lw=2, 
         label=f'SVM Souple (AUC = {auc_souple:.3f})')
plt.plot([0, 1], [0, 1], color='grey', lw=2, linestyle='--', label='Aléatoire (AUC = 0.500)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de Faux Positifs (FPR)')
plt.ylabel('Taux de Vrais Positifs (TPR)')
plt.title('Comparaison des Courbes ROC : SVM Rigide vs Souple')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

Fichier 'submission.csv' généré avec succès ! Vous
